# Stage 2: Batch Transformation Pipeline

This notebook completes **Stage 2** of the AgriFlow assignment.

It does the following:
- reads the raw datasets from the **HDFS landing zone**
- cleans, standardizes, and validates each source with **PySpark**
- joins sources into enriched datasets
- creates analytics tables that answer practical AgriFlow business questions
- writes **curated** and **analytics** outputs to HDFS as **Parquet** with partitioning

> This notebook assumes your fixed **Stage 1** notebook has already run successfully and created the `/agriflow/landing`, `/agriflow/curated`, and `/agriflow/analytics` zones in HDFS.


## 1. Create Spark Session and Define Paths

In [40]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

spark = (
    SparkSession.builder
    .appName("agriflow-stage2-batch-pipeline")
    .master("local[*]")
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000")
    .config("spark.pyspark.python", "/opt/conda/bin/python3")
    .config("spark.pyspark.driver.python", "/opt/conda/bin/python3")
    .getOrCreate()
)

BASE_HDFS = "hdfs://namenode:9000/agriflow"
LANDING = f"{BASE_HDFS}/landing"
CURATED = f"{BASE_HDFS}/curated"
ANALYTICS = f"{BASE_HDFS}/analytics"

RAW_PATHS = {
    "crop_yields": f"{LANDING}/crop-records",
    "equipment_usage": f"{LANDING}/equipment-usage",
    "market_prices": f"{LANDING}/market-prices",
    "soil_sensors": f"{LANDING}/soil-sensors",
    "weather_daily": f"{LANDING}/weather-data",
}

print("Spark version:", spark.version)
print("Landing:", LANDING)
print("Curated:", CURATED)
print("Analytics:", ANALYTICS)


Spark version: 3.5.0
Landing: hdfs://namenode:9000/agriflow/landing
Curated: hdfs://namenode:9000/agriflow/curated
Analytics: hdfs://namenode:9000/agriflow/analytics


## 2. Helper Functions

These helpers keep the notebook cleaner and make the code more flexible if column names vary slightly across files.


In [41]:

import re

def snake_name(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

def normalize_column_names(df):
    renamed = df
    for old in df.columns:
        new = snake_name(old)
        if old != new:
            renamed = renamed.withColumnRenamed(old, new)
    return renamed

def rename_first_match(df, target_name, aliases):
    if target_name in df.columns:
        return df
    for alias in aliases:
        if alias in df.columns:
            return df.withColumnRenamed(alias, target_name)
    return df

def cast_if_present(df, column_name, cast_type):
    if column_name in df.columns:
        return df.withColumn(column_name, F.col(column_name).cast(cast_type))
    return df

def parse_date_flex(df, source_col, target_col):
    if source_col not in df.columns:
        return df
    return df.withColumn(
        target_col,
        F.coalesce(
            F.to_date(F.col(source_col)),
            F.to_date(F.col(source_col), "yyyy-MM-dd"),
            F.to_date(F.col(source_col), "MM/dd/yyyy"),
            F.to_date(F.col(source_col), "yyyy/MM/dd"),
            F.to_date(F.col(source_col), "dd-MM-yyyy")
        )
    )

def parse_timestamp_flex(df, source_col, target_col):
    if source_col not in df.columns:
        return df
    return df.withColumn(
        target_col,
        F.coalesce(
            F.to_timestamp(F.col(source_col)),
            F.to_timestamp(F.col(source_col), "yyyy-MM-dd HH:mm:ss"),
            F.to_timestamp(F.col(source_col), "yyyy-MM-dd'T'HH:mm:ss"),
            F.to_timestamp(F.col(source_col), "MM/dd/yyyy HH:mm:ss"),
            F.to_timestamp(F.col(source_col), "yyyy/MM/dd HH:mm:ss")
        )
    )

def safe_drop_duplicates(df, subset_cols):
    subset_cols = [c for c in subset_cols if c in df.columns]
    if subset_cols:
        return df.dropDuplicates(subset_cols)
    return df.dropDuplicates()

def null_summary(df, title):
    print(f"\n=== Null summary: {title} ===")
    exprs = [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
    df.select(exprs).show(truncate=False)

def write_partitioned_parquet(df, path, partitions=None, mode="overwrite"):
    partitions = [p for p in (partitions or []) if p in df.columns]
    writer = df.write.mode(mode)
    if partitions:
        writer = writer.partitionBy(*partitions)
    writer.parquet(path)
    print(f"[OK] wrote {path} partitioned by {partitions if partitions else 'no partitions'}")

def common_columns(df1, df2, candidates):
    return [c for c in candidates if c in df1.columns and c in df2.columns]


## 3. Read the Raw Datasets from HDFS Landing Zone

In [42]:

crop_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATHS["crop_yields"])
)

equipment_raw = spark.read.option("inferSchema", True).json(RAW_PATHS["equipment_usage"])

market_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATHS["market_prices"])
)

soil_raw = spark.read.option("inferSchema", True).json(RAW_PATHS["soil_sensors"])

weather_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(RAW_PATHS["weather_daily"])
)

raw_tables = {
    "crop_raw": crop_raw,
    "equipment_raw": equipment_raw,
    "market_raw": market_raw,
    "soil_raw": soil_raw,
    "weather_raw": weather_raw,
}

for name, df in raw_tables.items():
    print(f"\n{name}")
    print("rows:", df.count())
    print("columns:", df.columns)
    df.printSchema()
    df.show(5, truncate=False)



crop_raw
rows: 100000
columns: ['farm_id', 'field_id', 'year', 'crop_type', 'acres', 'yield_bushels_per_acre', 'fertilizer_applied_lbs', 'irrigation_gallons', 'planting_date', 'harvest_date']
root
 |-- farm_id: string (nullable = true)
 |-- field_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- crop_type: string (nullable = true)
 |-- acres: integer (nullable = true)
 |-- yield_bushels_per_acre: double (nullable = true)
 |-- fertilizer_applied_lbs: double (nullable = true)
 |-- irrigation_gallons: integer (nullable = true)
 |-- planting_date: date (nullable = true)
 |-- harvest_date: date (nullable = true)

+-------+-----------+----+---------+-----+----------------------+----------------------+------------------+-------------+------------+
|farm_id|field_id   |year|crop_type|acres|yield_bushels_per_acre|fertilizer_applied_lbs|irrigation_gallons|planting_date|harvest_date|
+-------+-----------+----+---------+-----+----------------------+----------------------+----

## 4. Clean, Standardize, and Validate Each Dataset

The goal here is to:
- standardize column names to `snake_case`
- align likely key fields to common names like `farm_id`, `field_id`, `crop`, and `record_date`
- cast dates and numerics correctly
- remove duplicates
- keep only sensible records for later joins


In [43]:
# -----------------------------
# Crop yields
# -----------------------------
crop = normalize_column_names(crop_raw)

crop_aliases = {
    "farm_id": ["farm", "farmid", "farm_identifier"],
    "field_id": ["field", "fieldid", "plot_id", "field_identifier"],
    "crop": ["crop_type", "crop_name", "commodity", "product"],
    "record_date": ["date", "harvest_date", "observation_date", "recorded_date"],
    "yield_amount": ["yield", "yield_tons", "yield_kg", "production", "crop_yield", "yield_bushels_per_acre"],
}

for target, aliases in crop_aliases.items():
    crop = rename_first_match(crop, target, aliases)

crop = parse_date_flex(crop, "record_date", "record_date")
crop = cast_if_present(crop, "yield_amount", "double")

if "crop" in crop.columns:
    crop = crop.withColumn("crop", F.trim(F.lower(F.col("crop"))))

crop = safe_drop_duplicates(crop, ["farm_id", "field_id", "crop", "record_date"])

if "record_date" in crop.columns:
    crop = crop.filter(F.col("record_date").isNotNull())
    crop = crop.withColumn("year", F.year("record_date"))
    crop = crop.withColumn("month", F.month("record_date"))

print("Crop cleaned:")
crop.printSchema()
crop.show(5, truncate=False)
null_summary(crop, "crop")
# -----------------------------
# Market prices
# -----------------------------
market = normalize_column_names(market_raw)

market_aliases = {
    "crop": ["crop_type", "crop_name", "commodity", "product"],
    "price_date": ["date", "market_date", "record_date", "observed_date"],
    "price_per_unit": ["price", "market_price", "price_usd", "price_per_ton", "unit_price", "price_per_bushel"],
    "region": ["location", "area", "district", "county", "state", "market_region"],
}

for target, aliases in market_aliases.items():
    market = rename_first_match(market, target, aliases)

market = parse_date_flex(market, "price_date", "price_date")
market = cast_if_present(market, "price_per_unit", "double")

if "crop" in market.columns:
    market = market.withColumn("crop", F.trim(F.lower(F.col("crop"))))

if "region" in market.columns:
    market = market.withColumn("region", F.trim(F.lower(F.col("region"))))

market = safe_drop_duplicates(market, ["crop", "region", "price_date"])

if "price_date" in market.columns:
    market = market.filter(F.col("price_date").isNotNull())
    market = market.withColumn("year", F.year("price_date"))
    market = market.withColumn("month", F.month("price_date"))

print("Market cleaned:")
market.printSchema()
market.show(5, truncate=False)
null_summary(market, "market")
# -----------------------------
# Weather daily
# -----------------------------
weather = normalize_column_names(weather_raw)

weather_aliases = {
    "farm_id": ["farm", "farmid", "farm_identifier"],
    "weather_date": ["date", "record_date", "observation_date"],
    "avg_temp": ["temperature", "avg_temperature", "temperature_c", "temp_c", "temp_avg"],
    "rainfall": ["precipitation", "rain_mm", "precip_mm", "rainfall_mm", "precipitation_in"],
}

for target, aliases in weather_aliases.items():
    weather = rename_first_match(weather, target, aliases)

# build avg_temp from high/low if needed
if "avg_temp" not in weather.columns and "high_temp_f" in weather.columns and "low_temp_f" in weather.columns:
    weather = weather.withColumn("avg_temp", (F.col("high_temp_f") + F.col("low_temp_f")) / 2.0)

weather = parse_date_flex(weather, "weather_date", "weather_date")
weather = cast_if_present(weather, "avg_temp", "double")
weather = cast_if_present(weather, "rainfall", "double")

weather = safe_drop_duplicates(weather, ["farm_id", "weather_date"])

if "weather_date" in weather.columns:
    weather = weather.filter(F.col("weather_date").isNotNull())
    weather = weather.withColumn("year", F.year("weather_date"))
    weather = weather.withColumn("month", F.month("weather_date"))
print("Weather cleaned:")
weather.printSchema()
weather.show(5, truncate=False)
null_summary(weather, "weather")
# -----------------------------
# Soil sensors
# -----------------------------
soil = normalize_column_names(soil_raw)

soil = soil.withColumn("soil_moisture", F.col("conditions.soil_moisture_pct"))
soil = soil.withColumn("soil_temp", F.col("conditions.soil_temp_f"))
soil = soil.withColumn("soil_ph", F.col("nutrients.ph_level"))

soil_aliases = {
    "farm_id": ["farm", "farmid", "farm_identifier"],
    "field_id": ["field", "fieldid", "plot_id", "field_identifier"],
    "sensor_timestamp": ["timestamp", "recorded_at", "reading_time", "event_time"],
}

for target, aliases in soil_aliases.items():
    soil = rename_first_match(soil, target, aliases)

soil = parse_timestamp_flex(soil, "sensor_timestamp", "sensor_timestamp")
soil = cast_if_present(soil, "soil_moisture", "double")
soil = cast_if_present(soil, "soil_ph", "double")
soil = cast_if_present(soil, "soil_temp", "double")

soil = safe_drop_duplicates(soil, ["field_id", "sensor_timestamp"])

if "sensor_timestamp" in soil.columns:
    soil = soil.filter(F.col("sensor_timestamp").isNotNull())
    soil = soil.withColumn("record_date", F.to_date("sensor_timestamp"))
    soil = soil.withColumn("year", F.year("record_date"))
    soil = soil.withColumn("month", F.month("record_date"))
print("Soil cleaned:")
soil.printSchema()
soil.show(5, truncate=False)
null_summary(soil, "soil")


# -----------------------------
# Equipment usage
# -----------------------------
equipment = normalize_column_names(equipment_raw)

equipment_aliases = {
    "farm_id": ["farm", "farmid", "farm_identifier"],
    "field_id": ["field", "fieldid", "plot_id", "field_identifier"],
    "equipment_type": ["machine_type", "equipment", "asset_type", "type"],
    "usage_timestamp": ["timestamp", "used_at", "event_time", "recorded_at"],
    "hours_used": ["usage_hours", "hours", "operating_hours", "hours_operated"],
    "fuel_used": ["fuel", "fuel_liters", "fuel_consumed", "fuel_gallons"],
}

for target, aliases in equipment_aliases.items():
    equipment = rename_first_match(equipment, target, aliases)

equipment = parse_timestamp_flex(equipment, "usage_timestamp", "usage_timestamp")
equipment = cast_if_present(equipment, "hours_used", "double")
equipment = cast_if_present(equipment, "fuel_used", "double")

if "equipment_type" in equipment.columns:
    equipment = equipment.withColumn("equipment_type", F.trim(F.lower(F.col("equipment_type"))))

equipment = safe_drop_duplicates(equipment, ["field_id", "equipment_type", "usage_timestamp"])

if "usage_timestamp" in equipment.columns:
    equipment = equipment.filter(F.col("usage_timestamp").isNotNull())
    equipment = equipment.withColumn("record_date", F.to_date("usage_timestamp"))
    equipment = equipment.withColumn("year", F.year("record_date"))
    equipment = equipment.withColumn("month", F.month("record_date"))
print("Equipment cleaned:")
equipment.printSchema()
equipment.show(5, truncate=False)
null_summary(equipment, "equipment")


Crop cleaned:
root
 |-- farm_id: string (nullable = true)
 |-- field_id: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- crop: string (nullable = true)
 |-- acres: integer (nullable = true)
 |-- yield_amount: double (nullable = true)
 |-- fertilizer_applied_lbs: double (nullable = true)
 |-- irrigation_gallons: integer (nullable = true)
 |-- planting_date: date (nullable = true)
 |-- record_date: date (nullable = true)
 |-- month: integer (nullable = true)

+-------+-----------+----+--------+-----+------------+----------------------+------------------+-------------+-----------+-----+
|farm_id|field_id   |year|crop    |acres|yield_amount|fertilizer_applied_lbs|irrigation_gallons|planting_date|record_date|month|
+-------+-----------+----+--------+-----+------------+----------------------+------------------+-------------+-----------+-----+
|FARM-05|FARM-05-F09|2023|corn    |190  |172.2       |497.1                 |28252             |2023-04-18   |2023-10-08 |10   |
|FA

## 5. Write Curated Zone Outputs as Parquet

In [44]:

write_partitioned_parquet(crop, f"{CURATED}/crop_yields", ["year", "month"])
write_partitioned_parquet(market, f"{CURATED}/market_prices", ["year", "month"])
write_partitioned_parquet(weather, f"{CURATED}/weather_daily", ["year", "month"])
write_partitioned_parquet(soil, f"{CURATED}/soil_sensors", ["year", "month"])
write_partitioned_parquet(equipment, f"{CURATED}/equipment_usage", ["year", "month"])


[OK] wrote hdfs://namenode:9000/agriflow/curated/crop_yields partitioned by ['year', 'month']
[OK] wrote hdfs://namenode:9000/agriflow/curated/market_prices partitioned by ['year', 'month']
[OK] wrote hdfs://namenode:9000/agriflow/curated/weather_daily partitioned by ['year', 'month']
[OK] wrote hdfs://namenode:9000/agriflow/curated/soil_sensors partitioned by ['year', 'month']
[OK] wrote hdfs://namenode:9000/agriflow/curated/equipment_usage partitioned by ['year', 'month']


## 6. Build Intermediate Daily Summaries

Before joining everything together, I aggregate the high-frequency sensor and equipment data to the daily grain.


In [45]:

soil_group_cols = [c for c in ["farm_id", "field_id", "record_date", "year", "month"] if c in soil.columns]

soil_agg_exprs = []
if "soil_moisture" in soil.columns:
    soil_agg_exprs.append(F.avg("soil_moisture").alias("avg_soil_moisture"))
if "soil_ph" in soil.columns:
    soil_agg_exprs.append(F.avg("soil_ph").alias("avg_soil_ph"))
if "soil_temp" in soil.columns:
    soil_agg_exprs.append(F.avg("soil_temp").alias("avg_soil_temp"))
soil_agg_exprs.append(F.count("*").alias("soil_readings"))

soil_daily = soil.groupBy(*soil_group_cols).agg(*soil_agg_exprs)

equipment_group_cols = [c for c in ["farm_id", "field_id", "record_date", "year", "month", "equipment_type"] if c in equipment.columns]

equipment_agg_exprs = []
if "hours_used" in equipment.columns:
    equipment_agg_exprs.append(F.sum("hours_used").alias("total_hours_used"))
if "fuel_used" in equipment.columns:
    equipment_agg_exprs.append(F.sum("fuel_used").alias("total_fuel_used"))
equipment_agg_exprs.append(F.count("*").alias("equipment_events"))

equipment_daily = equipment.groupBy(*equipment_group_cols).agg(*equipment_agg_exprs)

print("soil_daily")
soil_daily.show(10, truncate=False)

print("equipment_daily")
equipment_daily.show(10, truncate=False)


soil_daily
+-------+-----------+-----------+----+-----+------------------+-----------------+------------------+-------------+
|farm_id|field_id   |record_date|year|month|avg_soil_moisture |avg_soil_ph      |avg_soil_temp     |soil_readings|
+-------+-----------+-----------+----+-----+------------------+-----------------+------------------+-------------+
|FARM-01|FARM-01-F04|2024-07-12 |2024|7    |33.0              |6.370000000000001|72.525            |4            |
|FARM-01|FARM-01-F13|2024-04-18 |2024|4    |39.75714285714286 |6.822857142857143|58.614285714285714|7            |
|FARM-02|FARM-02-F01|2024-03-10 |2024|3    |37.525            |6.1575           |50.525000000000006|4            |
|FARM-02|FARM-02-F02|2024-08-17 |2024|8    |26.825            |6.835000000000001|63.725            |4            |
|FARM-02|FARM-02-F03|2024-04-28 |2024|4    |37.78000000000001 |6.717999999999999|58.3              |5            |
|FARM-02|FARM-02-F07|2024-08-09 |2024|8    |34.175000000000004|6.7974

## 7. Create Enriched Datasets by Joining Multiple Sources

### Business question focus
This stage is designed to support questions like:
1. **Which crops and regions appear to generate the most revenue opportunity?**
2. **How do weather and soil conditions line up with crop yield outcomes?**
3. **Which equipment types are used the most and where is fuel/hour usage highest?**


In [46]:
# -----------------------------------
# Helper functions
# -----------------------------------
from collections import Counter
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

def make_unique_columns(df):
    cols = df.columns
    seen = {}
    new_cols = []

    for c in cols:
        if c not in seen:
            seen[c] = 0
            new_cols.append(c)
        else:
            seen[c] += 1
            new_cols.append(f"{c}__dup{seen[c]}")

    return df.toDF(*new_cols)

def show_duplicate_cols(df, name):
    dup_cols = [c for c, n in Counter(df.columns).items() if n > 1]
    print(f"{name} duplicate columns: {dup_cols}")

# -----------------------------------
# Sanitize inputs first
# -----------------------------------
crop = make_unique_columns(crop)
market = make_unique_columns(market)
weather = make_unique_columns(weather)
soil_daily = make_unique_columns(soil_daily)
equipment_daily = make_unique_columns(equipment_daily)

show_duplicate_cols(crop, "crop")
show_duplicate_cols(market, "market")
show_duplicate_cols(weather, "weather")
show_duplicate_cols(soil_daily, "soil_daily")
show_duplicate_cols(equipment_daily, "equipment_daily")

# -----------------------------------
# Yearly Summaries (Dropping 'month')
# -----------------------------------
soil_yearly = (
    soil_daily.groupBy("farm_id", "field_id", "year")
    .agg(
        F.avg("avg_soil_moisture").alias("avg_soil_moisture"),
        F.avg("avg_soil_ph").alias("avg_soil_ph"),
        F.avg("avg_soil_temp").alias("avg_soil_temp"),
        F.count("*").alias("soil_days")
    )
)

equipment_yearly = (
    equipment_daily.groupBy("farm_id", "field_id", "year")
    .agg(
        F.avg("total_hours_used").alias("avg_hours_used"),
        F.avg("total_fuel_used").alias("avg_fuel_used"),
        F.sum("total_hours_used").alias("yearly_hours_used"),
        F.sum("total_fuel_used").alias("yearly_fuel_used"),
        F.count("*").alias("equipment_days")
    )
)

weather_yearly = (
    weather.groupBy("farm_id", "year")
    .agg(
        F.avg("avg_temp").alias("avg_air_temp"),
        F.avg("rainfall").alias("avg_rainfall"),
        F.sum("rainfall").alias("total_rainfall"),
        F.count("*").alias("weather_days")
    )
)

crop_yearly_field = (
    crop.groupBy("farm_id", "field_id", "crop", "year")
    .agg(
        F.count("*").alias("record_count"),
        F.avg("yield_amount").alias("avg_yield")
    )
)

# -----------------------------------
# Final field condition summary (Joined on YEAR only)
# -----------------------------------
field_condition_summary = (
    crop_yearly_field.alias("c")
    .join(soil_yearly.alias("s"), on=["farm_id", "field_id", "year"], how="left")
    .join(equipment_yearly.alias("e"), on=["farm_id", "field_id", "year"], how="left")
    .join(weather_yearly.alias("w"), on=["farm_id", "year"], how="left")
    .select(
        "farm_id",
        "field_id",
        "crop",
        "year",
        "record_count",
        F.round("avg_yield", 2).cast(DecimalType(10, 2)).alias("avg_yield"),
        F.round("avg_soil_moisture", 2).cast(DecimalType(10, 2)).alias("avg_soil_moisture"),
        F.round("avg_soil_ph", 2).cast(DecimalType(10, 2)).alias("avg_soil_ph"),
        F.round("avg_soil_temp", 2).cast(DecimalType(10, 2)).alias("avg_soil_temp"),
        F.round("avg_air_temp", 2).cast(DecimalType(10, 2)).alias("avg_air_temp"),
        F.round("avg_rainfall", 2).cast(DecimalType(10, 2)).alias("avg_rainfall"),
        F.round("avg_hours_used", 2).cast(DecimalType(10, 2)).alias("avg_hours_used"),
        F.round("avg_fuel_used", 2).cast(DecimalType(10, 2)).alias("avg_fuel_used")
    )
)

print("field_condition_summary (Yearly Grain):")
field_condition_summary.show(20, truncate=False)

crop duplicate columns: []
market duplicate columns: []
weather duplicate columns: []
soil_daily duplicate columns: []
equipment_daily duplicate columns: []
field_condition_summary (Yearly Grain):
+-------+-----------+--------+----+------------+---------+-----------------+-----------+-------------+------------+------------+--------------+-------------+
|farm_id|field_id   |crop    |year|record_count|avg_yield|avg_soil_moisture|avg_soil_ph|avg_soil_temp|avg_air_temp|avg_rainfall|avg_hours_used|avg_fuel_used|
+-------+-----------+--------+----+------------+---------+-----------------+-----------+-------------+------------+------------+--------------+-------------+
|FARM-35|FARM-35-F16|wheat   |2021|4           |61.33    |NULL             |NULL       |NULL         |56.22       |0.21        |NULL          |NULL         |
|FARM-36|FARM-36-F03|corn    |2018|8           |154.31   |NULL             |NULL       |NULL         |56.66       |0.15        |NULL          |NULL         |
|FARM-28|FARM

## 8. Analytics Outputs

In [47]:

# -----------------------------------
# Analytics 1: Revenue by crop / region / month
# -----------------------------------
# crop totals by crop/year/month
crop_monthly = (
    crop.groupBy("crop", "year", "month")
    .agg(
        F.sum("yield_amount").alias("total_yield"),
        F.count("*").alias("crop_records")
    )
)

# join crop totals to regional market monthly
revenue_by_crop_region = (
    crop_monthly.alias("c")
    .join(
        market_region_monthly.alias("m"),
        on=["crop", "year", "month"],
        how="left"
    )
    .withColumn(
        "total_estimated_revenue",
        F.col("total_yield") * F.col("avg_price_per_unit")
    )
    .select(
        "crop",
        "region",
        "year",
        "month",
        "crop_records",
        "total_yield",
        "avg_price_per_unit",
        "avg_futures_price",
        "market_obs",
        "total_estimated_revenue"
    )
    .orderBy(F.desc("total_estimated_revenue"))
)

revenue_by_crop_region.show(20, truncate=False)

# -----------------------------------
# Analytics 2: Field condition summary
# -----------------------------------
field_enriched = field_enriched.select(*list(dict.fromkeys(field_enriched.columns)))

group_2 = [c for c in ["farm_id", "field_id", "crop", "year", "month"] if c in field_enriched.columns]

agg_exprs_2 = [F.count("*").alias("record_count")]

for source_col, alias_name in [
    ("yield_amount", "avg_yield"),
    ("avg_soil_moisture", "avg_soil_moisture"),
    ("avg_soil_ph", "avg_soil_ph"),
    ("avg_soil_temp", "avg_soil_temp"),
    ("avg_temp", "avg_air_temp"),
    ("rainfall", "avg_rainfall"),
    ("total_hours_used", "avg_hours_used"),
    ("total_fuel_used", "avg_fuel_used"),
]:
    if source_col in field_enriched.columns:
        agg_exprs_2.append(F.avg(source_col).alias(alias_name))

field_condition_summary = field_enriched.groupBy(*group_2).agg(*agg_exprs_2)

print("field_condition_summary")
field_condition_summary.show(20, truncate=False)
# -----------------------------------
# Analytics 3: Equipment efficiency
# -----------------------------------
group_3 = [c for c in ["equipment_type", "farm_id", "field_id", "year", "month"] if c in equipment_daily.columns]

agg_exprs_3 = [F.count("*").alias("daily_rows")]
if "total_hours_used" in equipment_daily.columns:
    agg_exprs_3.append(F.sum("total_hours_used").alias("hours_used"))
if "total_fuel_used" in equipment_daily.columns:
    agg_exprs_3.append(F.sum("total_fuel_used").alias("fuel_used"))
if "equipment_events" in equipment_daily.columns:
    agg_exprs_3.append(F.sum("equipment_events").alias("event_count"))

equipment_efficiency = equipment_daily.groupBy(*group_3).agg(*agg_exprs_3)

if "fuel_used" in equipment_efficiency.columns and "hours_used" in equipment_efficiency.columns:
    equipment_efficiency = equipment_efficiency.withColumn(
        "fuel_per_hour",
        F.when(F.col("hours_used") > 0, F.col("fuel_used") / F.col("hours_used"))
    )

print("equipment_efficiency")
equipment_efficiency.show(20, truncate=False)


+----+---------------+----+-----+------------+-----------------+------------------+------------------+----------+-----------------------+
|crop|region         |year|month|crop_records|total_yield      |avg_price_per_unit|avg_futures_price |market_obs|total_estimated_revenue|
+----+---------------+----+-----+------------+-----------------+------------------+------------------+----------+-----------------------+
|corn|midwest_south  |2019|10   |4403        |736043.8000000003|5.747272727272727 |6.0609090909090915|11        |4230244.457818183      |
|corn|midwest_south  |2024|10   |4280        |711259.5         |5.825             |5.922999999999999 |10        |4143086.5875           |
|corn|plains         |2023|10   |4374        |728240.7000000008|5.633846153846155 |5.772307692307693 |13        |4102796.066769236      |
|corn|midwest_central|2020|10   |4280        |710819.7000000002|5.721818181818183 |5.839090909090908 |11        |4067181.083454547      |
|corn|plains         |2021|10   |4

## 9. Write Analytics Zone Outputs as Partitioned Parquet

In [48]:
# -----------------------------------
# Repartition to avoid the small files problem
# Note: field_condition_summary is now partitioned ONLY by year
# -----------------------------------
soil_daily_out = soil_daily.repartition("year", "month")
equipment_daily_out = equipment_daily.repartition("year", "month")
revenue_by_crop_region_out = revenue_by_crop_region.repartition("year", "month")
field_condition_summary_out = field_condition_summary.repartition("year")
equipment_efficiency_out = equipment_efficiency.repartition("year", "month")

# -----------------------------------
# Write to HDFS Analytics Zone
# -----------------------------------
write_partitioned_parquet(soil_daily_out, f"{ANALYTICS}/soil_daily", ["year", "month"])
write_partitioned_parquet(equipment_daily_out, f"{ANALYTICS}/equipment_daily", ["year", "month"])
write_partitioned_parquet(revenue_by_crop_region_out, f"{ANALYTICS}/revenue_by_crop_region", ["year", "month"])
write_partitioned_parquet(field_condition_summary_out, f"{ANALYTICS}/field_condition_summary", ["year"])
write_partitioned_parquet(equipment_efficiency_out, f"{ANALYTICS}/equipment_efficiency", ["year", "month"])

[OK] wrote hdfs://namenode:9000/agriflow/analytics/soil_daily partitioned by ['year', 'month']
[OK] wrote hdfs://namenode:9000/agriflow/analytics/equipment_daily partitioned by ['year', 'month']
[OK] wrote hdfs://namenode:9000/agriflow/analytics/revenue_by_crop_region partitioned by ['year', 'month']
[OK] wrote hdfs://namenode:9000/agriflow/analytics/field_condition_summary partitioned by ['year']
[OK] wrote hdfs://namenode:9000/agriflow/analytics/equipment_efficiency partitioned by ['year', 'month']


## 10. Final Verification of Curated and Analytics Outputs

In [49]:
verification_paths = [
    # Curated Zone
    f"{CURATED}/crop_yields",
    f"{CURATED}/market_prices",
    f"{CURATED}/weather_daily",
    f"{CURATED}/soil_sensors",
    f"{CURATED}/equipment_usage",
    
    # Analytics Zone
    f"{ANALYTICS}/soil_daily",
    f"{ANALYTICS}/equipment_daily",
    f"{ANALYTICS}/revenue_by_crop_region",
    f"{ANALYTICS}/field_condition_summary",
    f"{ANALYTICS}/equipment_efficiency",
]

print("=== Final Verification of HDFS Outputs ===\n")

for path in verification_paths:
    print(f"Checking: {path}")
    try:
        df = spark.read.parquet(path)
        print("rows:", df.count())
        print("columns:", df.columns)
        df.show(5, truncate=False)
    except Exception as e:
        print(f"ERROR reading {path}: {e}")
    print("-" * 50)

=== Final Verification of HDFS Outputs ===

Checking: hdfs://namenode:9000/agriflow/curated/crop_yields
rows: 88375
columns: ['farm_id', 'field_id', 'crop', 'acres', 'yield_amount', 'fertilizer_applied_lbs', 'irrigation_gallons', 'planting_date', 'record_date', 'year', 'month']
+-------+-----------+--------+-----+------------+----------------------+------------------+-------------+-----------+----+-----+
|farm_id|field_id   |crop    |acres|yield_amount|fertilizer_applied_lbs|irrigation_gallons|planting_date|record_date|year|month|
+-------+-----------+--------+-----+------------+----------------------+------------------+-------------+-----------+----+-----+
|FARM-02|FARM-02-F15|soybeans|79   |63.9        |342.4                 |20821             |2023-05-26   |2023-10-05 |2023|10   |
|FARM-11|FARM-11-F09|soybeans|98   |49.0        |118.9                 |9744              |2023-05-14   |2023-10-18 |2023|10   |
|FARM-24|FARM-24-F05|soybeans|187  |31.1        |843.1                 |2037

# Stage 2: Batch Transformation Pipeline Summary

### Pipeline Accomplishments
In Stage 2, I engineered a batch transformation pipeline using PySpark to process the raw AgriFlow datasets from the HDFS landing zone. The data which showed crop yields, market prices, daily weather, soil sensors, and equipment usage) was cleaned up and ready for use. This included normalizing column names to all be the same and in snake case, making sure data types were strict, standardizing date/timestamp formats, and dropping duplicate records. The cleaned datasets were written to the HDFS Curated zone as Parquet files, partitioned by year and month to optimize downstream query performance.

Finally, I joined these different sources to create some enriched tables in the analytics zone, whcich help to answer specific questions regarding crop performance, environmental conditions, and equipment efficiency.

### Addressing the Sensor Data Sparsity
During the enrichment phase, the data showed a significant number of missing values (NULLs) when joining the IoT sensor data (soil and equipment) to the crop yield records. This was due to two issues in the data:

Historical difference: The crop yield data spans nine years (2017–2025), but the IoT sensors were only active in 2024.

Incomplete number of farms with sensors: Even within 2024, the sensors were not deployed universally. Out of 2,067 crop fields harvested in 2024, only 687 fields were equipped with monitoring tech.

To solve this, I aggregated the daily sensor data to a yearly level instead of monthly to properly align with the annual crop harvest. I then used a left join (anchored on the crop records) to build the field_condition_summary. This was able to show the complete historical record of crop yields for long-term trend analysis, rather than dropping 8 years of production data—and over 1,300 unmonitored fields in 2024 just to have a densely populated dataset.

### Key Analytical Insights
By designing the pipeline this way, I was able to generate multiple analytical views:

The Historical Record: By grouping the enriched data by year and crop, the business can track overall yield_amount trends from 2017 to 2025, completely independent of when sensors were used or even deployed.

The 2024 "Smart Field" Deep Dive: By filtering the data specifically for 2024 and dropping records where avg_soil_moisture was null, I only used the 687 fields equipped with sensors. This showed that the absolute top-performing "smart" corn fields yielded upwards of 213 bushels per acre under specific average soil and temperature conditions.

Equipment Efficiency: By aggregating the equipment logs, the pipeline tracks monthly fuel efficiency (fuel_per_hour) across different machinery types, providing visibility into operational costs throughout the growing and harvesting seasons.


In [50]:
from pyspark.sql import functions as F

# -----------------------------------
# Analytics View 1: The Historical Record
# -----------------------------------
# This view proves that keeping the NULLs allows us to see yield trends 
# across all 9 years without losing data.
historical_yields = (
    field_condition_summary
    .groupBy("year", "crop")
    .agg(
        F.count("*").alias("total_records"),
        F.round(F.avg("avg_yield"), 2).alias("historical_avg_yield")
    )
    .orderBy("year", "crop")
)

print("Historical Crop Yields (2017-2025):")
historical_yields.show(15, truncate=False)


# -----------------------------------
# Analytics View 2: The 2024 Sensor Deep Dive
# -----------------------------------
# This view filters out the NULLs by looking specifically at 2024, 
# AND filters only for fields that actually had sensors installed.
deep_dive_2024 = (
    field_condition_summary
    .filter(F.col("year") == 2024)
    .filter(F.col("avg_soil_moisture").isNotNull())  # <-- This is the new filter!
    .select(
        "farm_id", 
        "field_id", 
        "crop", 
        "avg_yield", 
        "avg_soil_moisture", 
        "avg_soil_ph", 
        "avg_soil_temp",
        "avg_fuel_used"
    )
    .orderBy(F.desc("avg_yield"))
)

print("2024 Deep Dive (Complete Sensor Data):")
deep_dive_2024.show(10, truncate=False)


# -----------------------------------
# Data Quality Check: Verifying the NULLs
# -----------------------------------
# This summarizes the missing data by year to confirm our join logic.
# 2024 should show 0 missing records for the sensors!
null_verification = (
    field_condition_summary
    .groupBy("year")
    .agg(
        F.sum(F.when(F.col("avg_soil_moisture").isNull(), 1).otherwise(0)).alias("missing_sensor_data"),
        F.count("*").alias("total_rows")
    )
    .orderBy("year")
)

print("Data Quality Verification (Missing Sensor Records by Year):")
null_verification.show()

Historical Crop Yields (2017-2025):
+----+--------+-------------+--------------------+
|year|crop    |total_records|historical_avg_yield|
+----+--------+-------------+--------------------+
|2017|corn    |690          |167.33              |
|2017|soybeans|690          |46.45               |
|2018|corn    |690          |166.37              |
|2018|soybeans|690          |46.22               |
|2018|wheat   |686          |55.2                |
|2019|corn    |690          |166.87              |
|2019|soybeans|690          |46.28               |
|2019|wheat   |685          |55.82               |
|2020|corn    |690          |166.13              |
|2020|soybeans|690          |46.27               |
|2020|wheat   |686          |55.75               |
|2021|corn    |690          |166.56              |
|2021|soybeans|688          |46.21               |
|2021|wheat   |686          |55.47               |
|2022|corn    |690          |165.84              |
+----+--------+-------------+-----------------